# Data Quality Findings

This notebook profiles the Silver layer and documents the main findings used to justify Gold governance KPIs.

In [1]:
from pathlib import Path
import ast
import json

import pandas as pd

silver_dir = Path('../datalake_silver')
files = sorted(silver_dir.glob('*.parquet'))
frames = [pd.read_parquet(path) for path in files]
silver = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
silver.head()

,newsLink,comments,_id,snapshot_id,snapshot_date,type,id,url,twitterUrl,text,...,quoted_tweet_quoted_tweet_retweeted_tweet,quoted_tweet_quoted_tweet_isLimitedReply,quoted_tweet_quoted_tweet_communityInfo,quoted_tweet_quoted_tweet_article,author_profile_bio_entities_description_symbols,entities_timestamps,quoted_tweet_author_profile_bio_entities_description_hashtags,quoted_tweet_author_profile_bio_entities_url_urls,text_cleaned,text_length
0,https://www.gsmarena.com/xiaomi_unveils_smart_...,"[""I like how at least some still make earbuds ...",6a19f676d5d72db4fa2cf569,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
1,https://www.gsmarena.com/samsung_is_now_shippi...,"[""Ai sux"", ""Ai will ruin this planet"", ""AI wil...",6a19f676d5d72db4fa2cf56a,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2,https://www.gsmarena.com/samsung_galaxy_z_fold...,"[""Waiting for the day Samsung increases batter...",6a19f676d5d72db4fa2cf56b,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
3,https://www.gsmarena.com/vertu_unveils_alphafo...,"[""Rebranded zte with leather slapped onto it. ...",6a19f676d5d72db4fa2cf56c,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
4,https://www.gsmarena.com/hmd_grand_renders_and...,"[""-, 28 May 20261. How did you get banned in u...",6a19f676d5d72db4fa2cf56d,6a19f676d5d72db4fa2cf568,1780086390412,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN


In [2]:
if not silver.empty:
    null_rates = silver.isna().mean().sort_values(ascending=False).to_frame('null_rate')
    display(null_rates.head(15))

    numeric_cols = silver.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        describe = silver[numeric_cols].describe().T
        display(describe)

    if 'text' in silver.columns:
        text_lengths = silver['text'].astype(str).str.len()
        display(text_lengths.describe())
else:
    print('No Silver data found.')

,null_rate
inReplyToId,1.0
author_verifiedType,1.0
author_hasCustomTimelines,1.0
quoted_tweet_communityInfo,1.0
quoted_tweet_article,1.0
quoted_tweet_retweeted_tweet,1.0
quoted_tweet_quoted_tweet,1.0
quoted_tweet_author_verifiedType,1.0
quoted_tweet_inReplyToUsername,1.0
quoted_tweet_inReplyToId,1.0


,count,mean,std,min,25%,50%,75%,max
retweetCount,360.0,2.377778,8.388316e+00,0.0,0.0,0.0,0.0,53.0
replyCount,360.0,4.366667,1.714477e+01,0.0,0.0,0.0,1.0,132.0
likeCount,360.0,80.286111,3.357188e+02,0.0,0.0,0.0,3.0,2013.0
quoteCount,360.0,0.250000,1.332579e+00,0.0,0.0,0.0,0.0,11.0
viewCount,360.0,5663.419444,2.497630e+04,1.0,21.0,44.0,258.0,152549.0
bookmarkCount,360.0,5.619444,2.425439e+01,0.0,0.0,0.0,0.0,156.0
author_followers,360.0,10162.469444,4.580268e+04,2.0,114.0,471.0,2733.0,534221.0
author_following,360.0,1097.011111,1.825147e+03,0.0,190.0,431.0,992.0,14311.0
author_fastFollowersCount,360.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0
author_favouritesCount,360.0,47581.777778,7.493962e+04,0.0,1155.0,13050.0,56557.0,423507.0


count     360.000000
mean      260.697222
std       187.641868
min        61.000000
25%       146.000000
50%       242.000000
75%       284.000000
max      1070.000000
Name: text, dtype: float64

In [3]:
findings = []
if not silver.empty:
    if 'lang' in silver.columns:
        findings.append(('language_distribution', silver['lang'].value_counts(dropna=False).head(10)))
    if 'comments' in silver.columns:
        def count_comments(value):
            if isinstance(value, list):
                return len(value)
            if isinstance(value, str) and value.startswith('[') and value.endswith(']'):
                try:
                    parsed = ast.literal_eval(value)
                    return len(parsed) if isinstance(parsed, list) else 0
                except (ValueError, SyntaxError):
                    return 0
            return 0

        comments = silver['comments'].apply(count_comments)
        findings.append(('comment_count', comments.describe()))

    for title, value in findings:
        print(f'## {title}')
        display(value)
else:
    print('No findings generated.')

## language_distribution


lang
en     343
NaN     48
in      11
tl       5
und      1
Name: count, dtype: int64

## comment_count


count    408.000000
mean       2.455882
std       11.282524
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      151.000000
Name: comments, dtype: float64